# Notebook 02 — Operator KB & Paper Ingestion

**Goal**:
1. Load the FASTEXPR operator knowledge base from YAML and index it into Chroma
2. Ingest research papers / notes into the `papers` Chroma collection
3. Demonstrate RAG retrieval across all knowledge sources

**Requires**: Notebook 01 completed (vector store initialized)

In [1]:
import nest_asyncio
nest_asyncio.apply()

from alpha_agent.knowledge.vector_store import VectorStore

store = VectorStore()
print("Vector store loaded")

/Users/weiyanguang/vscodeprojects/WorldQuant-Brain-Alpha/myenv/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


Vector store loaded


## 1. Load and index FASTEXPR operators

In [2]:
from alpha_agent.data.operator_kb import OperatorKB

kb = OperatorKB()
print(f"Loaded {len(kb.operators)} operators from YAML")

# Index into Chroma
n_indexed = kb.index_into_store(store, force_refresh=False)
print(f"Indexed {n_indexed} operators (0 = already up to date)")

# Show categories
from collections import Counter
cats = Counter(op['category'] for op in kb.operators)
for cat, count in sorted(cats.items()):
    print(f"  {cat}: {count}")

Loaded 36 operators from YAML
Indexed 0 operators (0 = already up to date)
  conditional: 2
  cross_section: 10
  group: 3
  math: 5
  time_series: 16


In [3]:
# Search operators by research concept
from alpha_agent.knowledge.vector_store import COLLECTION_OPERATORS

query = "rolling correlation between two signals"
results = store.query(COLLECTION_OPERATORS, query, k=4)

print(f"Operator search: '{query}'\n")
for r in results:
    print(f"  [{r['metadata']['name']}] dist={r['distance']:.3f}")
    print(f"  {r['document'][:200]}\n")

Operator search: 'rolling correlation between two signals'

  [correlation] dist=0.517
  Operator: correlation
Category: time_series
Signature: correlation(x, y, d)
Returns: MATRIX
Description: Alias for ts_corr. Rolling correlation between x and y.
Example: correlation(close, volume, 10)

  [ts_corr] dist=0.532
  Operator: ts_corr
Category: time_series
Signature: ts_corr(x, y, d)
Returns: MATRIX
Description: Rolling correlation between x and y over d days.
Example: ts_corr(volume, returns, 10)
Tips: Captures d

  [mean] dist=0.737
  Operator: mean
Category: time_series
Signature: mean(x, d)
Returns: MATRIX
Description: Alias for ts_mean. Rolling mean of x over d days.
Example: mean(volume, 20)
Tips: Identical to ts_mean; prefer t

  [ts_sum] dist=0.746
  Operator: ts_sum
Category: time_series
Signature: ts_sum(x, d)
Returns: MATRIX
Description: Rolling sum of x over d days.
Example: ts_sum(volume, 5)
Tips: Aggregates over short windows for momentum-li



In [4]:
# Print compact prompt-ready operator reference
print(kb.to_prompt_text(categories=['time_series', 'cross_section']))

FASTEXPR Operator Reference:
  rank(x) → MATRIX: Cross-sectional rank of x, normalized to [0, 1].
  zscore(x) → MATRIX: Cross-sectional z-score normalization of x.
  scale(x) → MATRIX: Scales x so the sum of absolute values equals 1 cross-sectionally.
  demean(x) → MATRIX: Subtract the cross-sectional mean from x.
  group_rank(x, group) → MATRIX: Rank x within each group (e.g., subindustry, industry, sector).
  group_mean(x, weight, group) → MATRIX: Weighted mean of x within each group.
  group_neutralize(x, group) → MATRIX: Remove group mean from x (demeaned within group).
  vector_neut(x, y) → MATRIX: Orthogonalize x with respect to y (remove the component of x explained by y).
  regression_neut(x, y) → MATRIX: Residual of regressing x on y cross-sectionally.
  bucket(x, range='lo,hi,step') → MATRIX: Assigns x into quantile buckets defined by a range string.
  ts_mean(x, d) → MATRIX: Rolling mean of x over the past d days.
  ts_std_dev(x, d) → MATRIX: Rolling standard deviation of x 

## 2. Ingest research papers / notes

Place PDF or Markdown files in `data/papers/` and run the cells below.
You can also ingest plain-text snippets directly.

In [5]:
from alpha_agent.knowledge.paper_ingest import PaperIngest

pi = PaperIngest(store)

# Example: ingest a known factor insight as raw text
example_text = """
Cross-sectional momentum in equities (Jegadeesh & Titman 1993):
Stocks that performed well over the past 6-12 months tend to continue performing well
over the next 3-6 months (momentum). Conversely, short-term (1-month) returns
tend to reverse (short-term reversal). Momentum is stronger for:
- High volume, high liquidity stocks
- Stocks with analyst coverage
- Mid-cap stocks (less crowding vs. large-cap)

Volatility-adjusted momentum (Daniel & Moskowitz 2016):
Scale momentum exposure by 1/variance. This dramatically improves risk-adjusted returns
by reducing exposure during momentum crashes (typically high volatility periods).
FASTEXPR implementation: rank(ts_mean(returns, 252)) * (1 / ts_std_dev(returns, 60))

Earnings momentum / PEAD (Bernard & Thomas 1989):
Post-earnings announcement drift — stocks with positive earnings surprises continue
to drift up for several months. Can be proxied by analyst revision momentum.
"""

n = pi.ingest_text(example_text, source="momentum_notes")
print(f"Ingested {n} chunks from example text")

Ingested 1 chunks from example text


In [6]:
# Optionally: ingest all PDFs from a directory
import pathlib

papers_dir = pathlib.Path("../data/papers")
if papers_dir.exists():
    results = pi.ingest_directory(papers_dir)
    for fname, count in results.items():
        print(f"  {fname}: {count} chunks")
else:
    print(f"No papers directory found at {papers_dir}. Create it and add PDFs/MDs.")

No papers directory found at ../data/papers. Create it and add PDFs/MDs.


## 3. Cross-collection RAG demo

In [7]:
# Query across all collections simultaneously
query = "volatility scaling for momentum"
multi_results = store.query_multi(query, k_per_col=3)

print(f"Multi-collection search: '{query}'\n")
for r in multi_results[:9]:
    coll = r.get('collection', '?')
    dist = r.get('distance', 0)
    doc = r.get('document', '')[:150]
    print(f"[{coll}] dist={dist:.3f}")
    print(f"  {doc}")
    print()

Multi-collection search: 'volatility scaling for momentum'

[papers] dist=0.630
  Cross-sectional momentum in equities (Jegadeesh & Titman 1993): Stocks that performed well over the past 6-12 months tend to continue performing well 

[operators] dist=0.757
  Operator: scale
Category: cross_section
Signature: scale(x)
Returns: MATRIX
Description: Scales x so the sum of absolute values equals 1 cross-section

[operators] dist=0.766
  Operator: delta
Category: time_series
Signature: delta(x, d)
Returns: MATRIX
Description: x - delay(x, d). Change over d days.
Example: delta(close, 5

[operators] dist=0.767
  Operator: ts_std_dev
Category: time_series
Signature: ts_std_dev(x, d)
Returns: MATRIX
Description: Rolling standard deviation of x over d days.
Examp

[datafields] dist=0.807
  volume

[datafields] dist=0.824
  assets_curr

[datafields] dist=0.849
  low



## ✅ Notebook 02 Complete

You now have all knowledge indexed:
- Data fields (from NB01)
- FASTEXPR operators
- Research papers / notes

**Next**: Run `03_idea_to_expression_demo.ipynb` to see the full idea→expression pipeline without WQB.